### WRS-COVERAGE execution with Dask
This notebook demonstrates how to execute the WRS-COVERAGE application package using Calrissian on Kubernetes with task execution delegated to a Dask cluster.

Prepare a clean runtime environment by removing previous runs and results, then creating a fresh working directory.

In [1]:
%%bash
rm -rf runs/
rm -rf /calrissian/results/*
mkdir runs/

Copy the CWL application package into the temporary runtime folder used for execution.

In [14]:
%%bash
cp /workspace/app-wrs-coverage/cwl-workflow/app-wrs-coverage.cwl runs/

To allow Calrissian to locate the application package, copy it into the `/calrissian` mounted volume, which is shared with the Calrissian container.

In [15]:
%%bash
cp runs/app-wrs-coverage.cwl /calrissian/app-wrs-coverage.cwl

Inspect the Kubernetes Job manifest that will be submitted to the cluster.

In [ ]:
%%bash
cat /workspace/app-wrs-coverage/practice-lab/k8s-job.yaml

apiVersion: batch/v1
kind: Job
metadata:
  name: wrs-coverage
spec:
  ttlSecondsAfterFinished: 5
  template:
    spec:
      serviceAccountName: calrissian-sa
      containers:
        - name: calrissian
          image: calrissian:0.19.0-fix
          command: ["calrissian"]
          args:
            - --debug
            - --no-match-user
            - --stdout 
            - /calrissian/results.json
            - --stderr 
            - /calrissian/app.log
            - --max-ram 
            - 48G
            - --max-cores 
            - "8"
            - --tmp-outdir-prefix 
            - /calrissian/tmp/ 
            - --outdir
            - /calrissian/results
            - --usage-report 
            - /calrissian/usage.json
            - --tool-logs-basepath 
            - /calrissian/logs
            - --dask-gateway-url
            - "http://traefik-dask-gateway.eoap-dask-gateway.svc.cluster.local:80"
            - /calrissian/app-wrs-coverage.cwl#wrs-coverage
          vo

Use `kubectl` to apply the Kubernetes Job definition and start the execution.

In [ ]:
%%bash
kubectl apply -f /workspace/app-wrs-coverage/practice-lab/k8s-job.yaml

job.batch/wrs-coverage created


Block execution until the Kubernetes Job completes or the timeout is reached.

In [7]:
%%bash
kubectl wait --for=condition=complete --timeout=600s job/wrs-coverage

job.batch/wrs-coverage condition met


List the generated output files and preview the resulting visualization.

In [12]:
%%bash
ls -lt /calrissian/results/

total 2752
-rw-r--r-- 1 root root 1818581 Jan 14 10:26 acq-by-wrs-tile.parquet
-rw-r--r-- 1 root root  995471 Jan 14 10:26 acq-by-wrs-tile.png


In [13]:
from IPython.display import Image, display

display(Image(filename='/calrissian/results/acq-by-wrs-tile.png'))

<IPython.core.display.Image object>